Concept: Using OpenAI Python Client with Multiple LLM Providers

1. Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

2. Standard Implementation (OpenAI)

The baseline using GPT-4o-mini.

In [3]:
from turtle import mode
from httpx2 import Response
from openai.types.shared import response_format_json_object


client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Explain latency in 10 words."
    }]
)

print(f"OpenAI: {response.choices[0].message.content}")

OpenAI: Latency is the delay before data transmission begins or completes.


3. Architectural Summary: The Provider Factory

In a real-wrold application, we warp this in a factory funcation to maintain a single interface.

In [9]:
import select
from pydantic import config


def get_client(provider):
    """ 
        Resolve the correct OpenAI client and model based on the provider name.
    """

    config ={
        "openai": {"api_key": os.getenv("OPENAI_API_KEY"),
                    "base_url": None,
                    "model": "gpt-4o-mini"}
    }

    selected = config.get(provider.lower())
    if not selected:
        raise ValueError(f"Provider {provider} not supported.")
    
    return OpenAI(api_key=selected["api_key"], base_url=selected["base_url"]), selected["model"]


4. Architectural Summary: The Provider Factory
In a real-world application, we wrap this in a factory function to maintain a single interface.

In [10]:
def get_client(provider):
    """
    Resolves the correct OpenAI client and model based on the provider name.
    """
    config = {
        "openai": {"api_key": os.getenv("OPENAI_API_KEY"), "base_url": None, "model": "gpt-4o-mini"},
        # "deepseek": {"api_key": os.getenv("DEEPSEEK_API_KEY"), "base_url": "https://api.deepseek.com", "model": "deepseek-chat"},
        # "xai": {"api_key": os.getenv("XAI_API_KEY"), "base_url": "https://api.x.ai/v1", "model": "grok-4-1-fast-non-reasoning"}
    }
    
    selected = config.get(provider.lower())
    if not selected:
        raise ValueError(f"Provider {provider} not supported.")

    return OpenAI(api_key=selected["api_key"], base_url=selected["base_url"]), selected["model"]

5. Dynamic Provider Switching (The Switchboard)

This final funcation act as a unified entry point. You can now call any LLM for the same query by simply changing the provider name.

In [11]:
def chat_with_llm(provider_name, user_query):
    """
    A unified entry point to switch LLMs dynamically.
    """
    client, model = get_client(provider_name)
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_query}]
    )
    
    return response.choices[0].message.content

# Architects can now execute the same query across different brains:
query = "What is the main benefit of using a unified API client?"

print(f"--- Executing via OpenAI ---\n{chat_with_llm('openai', query)}\n")

--- Executing via OpenAI ---
The main benefit of using a unified API client is that it simplifies the integration process with multiple APIs by providing a consistent and standardized interface. This abstraction helps developers streamline their workflows, as they can interact with various services using the same set of methods and principles, regardless of the underlying differences in the APIs. 

Other key benefits include:

1. **Reduced Complexity**: A unified API client handles the complexities of dealing with different API versions, authentication methods, and protocols, allowing developers to focus on building their applications.

2. **Improved Efficiency**: Developers can save time by writing less code and reducing the need for extensive error handling related to different API implementations.

3. **Consistency**: A unified client can provide consistent error handling, response formats, and functionality across different APIs, making it easier for developers to understand and us